# Stable Diffusion generation and DAAM attention visualization experiments.

Research notebook: see the repository README before running. Saved outputs were cleared and embedded credentials removed. Dataset paths and dependency setup require adaptation.


In [ ]:

!git clone https://github.com/castorini/daam.git
%cd daam


!pip install torch torchvision ftfy matplotlib
!pip install diffusers==0.16.1 transformers==4.27.4 accelerate==0.18.0 jax jaxlib huggingface_hub


In [ ]:
!pip install daam

In [ ]:
from huggingface_hub import login
from daam import trace, set_seed
from diffusers import DiffusionPipeline
from matplotlib import pyplot as plt
import torch


print("Logging into Hugging Face...")

login()  # Enter your own token through the interactive prompt.
print("Hugging Face...login done ")

print("Checking GPU availability...")
is_cuda_available = torch.cuda.is_available()
print("GPU Available:", is_cuda_available)

In [ ]:


#model 1
print("Loading the model...")
model_id = "runwayml/stable-diffusion-v1-5"
device = 'cuda' if is_cuda_available else 'cpu'


if is_cuda_available:
    pipe = DiffusionPipeline.from_pretrained(model_id, use_auth_token=True, torch_dtype=torch.float16, use_safetensors=True, variant='fp16')
else:
    pipe = DiffusionPipeline.from_pretrained(model_id, use_auth_token=True)

pipe = pipe.to(device)
print("Model loaded.")

#prompt function
def run_prompt(pipe, prompt):
    print("Running the model...")
    result = pipe(prompt).images[0]
    plt.imshow(result)
    plt.axis('off')
    plt.show()
    print("Model run complete.")

# prompt
prompt = 'A dog runs across the field'
run_prompt(pipe, prompt)


In [ ]:
from diffusers import StableDiffusionPipeline
import daam

In [ ]:
#Model 2
model = StableDiffusionPipeline.from_pretrained('stabilityai/stable-diffusion-2-base')
model = model.to('cuda')

In [ ]:
from matplotlib import pyplot as plt
import numpy as np

def make_im_subplots(*args):
  fig, ax = plt.subplots(*args)

  for ax_ in ax.flatten():
    ax_.set_xticks([])
    ax_.set_yticks([])

  return fig, ax

prompt = 'An angry, bald man writing papers'

# Trace through generation
with daam.trace(model) as trc:
  output_image = model(prompt, num_inference_steps=20).images[0]
  global_heat_map = trc.compute_global_heat_map()

angry_heat_map = global_heat_map.compute_word_heat_map('angry')
bald_heat_map = global_heat_map.compute_word_heat_map('bald')
papers_heat_map = global_heat_map.compute_word_heat_map('papers')

plt.rcParams['figure.figsize'] = (8, 8)
fig, ax = make_im_subplots(2, 2)

# Original image
ax[0, 0].imshow(output_image)

# Angry heat map
angry_heat_map.plot_overlay(output_image, ax=ax[0, 1])

# Bald heat map
bald_heat_map.plot_overlay(output_image, ax=ax[1, 0])

# Papers heat map
papers_heat_map.plot_overlay(output_image, ax=ax[1, 1])

plt.show()

In [ ]:
prompt = 'A cute cat pouncing on a ball'
fig, ax = make_im_subplots(4, 4)

# Trace through generation
with daam.trace(model) as trc:
  output_image = model(prompt, num_inference_steps=20).images[0]

  for head_idx in range(16):
    ax_ = ax[head_idx // 4, head_idx % 4]
    global_heat_map = trc.compute_global_heat_map(head_idx=head_idx, layer_idx=2)
    angry_heat_map = global_heat_map.compute_word_heat_map('cat')
    angry_heat_map.plot_overlay(output_image, ax=ax_)
    ax_.set_title(f'Head {head_idx}')

plt.show()

In [ ]:
# Trace through generation
import time
import torch
import random

# Step 2: Verify GPU Availability
print("Checking GPU availability...")
is_cuda_available = torch.cuda.is_available()
print("GPU Available:", is_cuda_available)

# Step 3: Load the Model
print("Loading the model...")
#model_id = "runwayml/stable-diffusion-v1-5"
model_id = "stabilityai/stable-diffusion-2-base"
device = 'cuda' if is_cuda_available else 'cpu'

# Remove `torch_dtype=torch.float16` if no GPU is available
if is_cuda_available:
    pipe = DiffusionPipeline.from_pretrained(model_id, use_auth_token=True, torch_dtype=torch.float16, use_safetensors=True, variant='fp16')
else:
    pipe = DiffusionPipeline.from_pretrained(model_id, use_auth_token=True)

pipe = pipe.to(device)
print("Model loaded.")

# Function to set seed
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Generate images with different prompts
s = int(time.time())
set_seed(s)

with trace(pipe, save_heads=True) as trc:
    blue_image = pipe('a blue car driving down the street', num_inference_steps=20).images[0]
    blue_map = trc.compute_global_heat_map().compute_word_heat_map('blue')

set_seed(s)

with trace(pipe, load_heads=True) as trc:
    green_image = pipe('a green car driving down the street', num_inference_steps=20).images[0]
    green_map = trc.compute_global_heat_map().compute_word_heat_map('green')

set_seed(s)

with trace(pipe, load_heads=True) as trc:
    red_image = pipe('a red car driving down the street', num_inference_steps=20).images[0]
    red_map = trc.compute_global_heat_map().compute_word_heat_map('red')

plt.rcParams['figure.figsize'] = (8, 8)
fig, ax = plt.subplots(2, 2)

# Original images
ax[0, 1].imshow(blue_image)
ax[1, 0].imshow(green_image)
ax[1, 1].imshow(red_image)

# Heat map
blue_map.plot_overlay(blue_image, ax=ax[0, 0])

plt.show()
print("Model run complete.")